## Importing

In [1]:
from datasets import load_dataset
import re
import sentencepiece as spm
import os
import torch
import random
import json
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/SLM'

Mounted at /content/drive


# Importing Datasets

In [ ]:
df = load_dataset("roneneldan/TinyStories", split = "train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
df = df.shuffle(seed=42)
split = df.train_test_split(test_size=0.2)

train = split["train"]
temp = split["test"]

temp = temp.shuffle(seed=42)
splitt = temp.train_test_split(test_size=0.5)

validation = splitt["train"]
test = splitt["test"]

In [ ]:
print("Train: ", train, "\nValidation: ",validation, "\nTest: ", test)

Train:  Dataset({
    features: ['text'],
    num_rows: 1695775
}) 
Validation:  Dataset({
    features: ['text'],
    num_rows: 211972
}) 
Test:  Dataset({
    features: ['text'],
    num_rows: 211972
})


In [ ]:
trainlst = train['text']
validationlst = validation['text']
testlst = test['text']

In [ ]:
dd = load_dataset("DeepPavlov/daily_dialog")

In [ ]:
# Assuming your DatasetDict is stored in a variable named 'dataset'

flattened_dialogues = {}

for split in dd.keys():
    # This loops through every row's dialogue list and flattens them into one single list of strings
    flattened_dialogues[split] = [
        utterance
        for dialogue_list in dd[split]['dialog']
        for utterance in dialogue_list
    ]

# --- Verification ---
for split, dialogues in flattened_dialogues.items():
    print(f"{split.capitalize()} individual utterances: {len(dialogues)}")


Train individual utterances: 474601
Validation individual utterances: 44126
Test individual utterances: 41206


In [ ]:
print("\nFirst 3 training utterances:")
print(flattened_dialogues['validation'][:3])


First 3 training utterances:
['Hello . Capital Hotel . May I help you ? ', ' Yes , unlikely my flight will be 2 hours due to the fog . Would you please keep my reservation ? ', ' Sure . May I have your name please ? ']


# Cleaning Dataset

In [ ]:
def cleans(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.!?;:\'\"()\-\n]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
dtst = folder_path + "/Datasets"

def append_jsonl(the_file, lst):
    i = 0
    with open(dtst + f'/{the_file}.jsonl', "a", encoding="utf-8") as f:
        for article in lst:
            temp = cleans(article)
            i+=1
            if i % 50000 == 0:
              print(f"Processed {i:,} articles")
            record = {"text": temp}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")





# from itertools import islice

# # =========================
# # Regexes (compile once)
# # =========================

# URL_RE = re.compile(r'http\S+|www\S+')
# SPECIAL_RE = re.compile(r'[^a-zA-Z0-9\s.!?;:\'\"()\-\n]')
# WHITESPACE_RE = re.compile(r'\s+')

# # =========================
# # Cleaning
# # =========================

# def clean_article(text):
#     text = URL_RE.sub('', text)
#     text = SPECIAL_RE.sub('', text)
#     text = WHITESPACE_RE.sub(' ', text)
#     return text.strip()

# # =========================
# # Streaming Dataset
# # =========================

# dataset = load_dataset(
#     "wikimedia/wikipedia",
#     "20231101.en",
#     split="train",
#     streaming=True
# )

# # =========================
# # Output files
# # =========================

# train_path = os.path.join(dtst, "train.jsonl")
# val_path = os.path.join(dtst, "val.jsonl")
# test_path = os.path.join(dtst, "test.jsonl")

# train_file = open(train_path, "a", encoding="utf-8")
# val_file = open(val_path, "a", encoding="utf-8")
# test_file = open(test_path, "a", encoding="utf-8")

# try:
#     for i, row in enumerate(islice(dataset, 300_000), start=1):

#         cleaned = clean_article(row["text"])

#         if not cleaned:
#             continue

#         record = json.dumps(
#             {"text": cleaned},
#             ensure_ascii=False
#         ) + "\n"

#         r = i % 100

#         if r < 90:
#             train_file.write(record)
#         elif r < 95:
#             val_file.write(record)
#         else:
#             test_file.write(record)

#         if i % 10000 == 0:
#             print(f"Processed {i:,} articles")

# except KeyboardInterrupt:
#     print("Stopped by user")

# finally:
#     train_file.close()
#     val_file.close()
#     test_file.close()


In [ ]:
append_jsonl('train',flattened_dialogues['train'])
append_jsonl('test',flattened_dialogues['test'])
append_jsonl('val', flattened_dialogues['validation'])

Processed 50,000 articles
Processed 100,000 articles
Processed 150,000 articles
Processed 200,000 articles
Processed 250,000 articles
Processed 300,000 articles
Processed 350,000 articles
Processed 400,000 articles
Processed 450,000 articles


# Tokenization

In [ ]:
#Creating my Own Tokenizer based on my data (Run only once, don't run a lot)
drive_model = folder_path + '/my_tokenizer'
dtst = folder_path + '/Datasets'

spm.SentencePieceTrainer.train(
        input = dtst + '/train.jsonl',
        model_prefix = folder_path,
        vocab_size=8000,
        model_type='bpe',
        pad_id=0,
        unk_id=1,
        bos_id=-1,
        eos_id=-1
    )

In [ ]:
EOS_TOKEN = 99

def stream_jsonl_texts(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            text = (obj.get("text") or "").strip()
            if text:
                yield text

def tokenize_and_save(split_name):
    sp = spm.SentencePieceProcessor(model_file=folder_path + "/my_tokenizer.model")
    path = os.path.join(folder_path,"Datasets",f"{split_name}.jsonl")
    out_path = os.path.join(folder_path,"tokenized",f"{split_name}.bin")
    with open(out_path, "wb") as fout:
        for i, text in enumerate(stream_jsonl_texts(path), 1):
            ids = sp.encode(text, out_type=int)
            ids.append(EOS_TOKEN)
            arr = np.array(ids, dtype=np.uint16)
            arr.tofile(fout)
            if i % 50000 == 0:
                print(f"Tokenized {i:,} articles")
    print(f"Finished {split_name}")

def run_pipeline():

    tokenize_and_save("train")
    tokenize_and_save("val")
    tokenize_and_save("test")


# def tokenize_and_save(split_name):
#     i=0
#     sp = spm.SentencePieceProcessor(model_file = folder_path + '/my_tokenizer.model')

#     path = os.path.join(folder_path, "Datasets", f"{split_name}.jsonl")
#     texts = load_jsonl_texts(path)

#     token_ids = []

#     for text in texts:
#         ids = sp.encode(text, out_type=int)
#         token_ids.extend(ids)
#         token_ids.append(EOS_TOKEN)
#         i+=1
#         if i % 50000 == 0:
#           print(f"Tokenized {i:,} articles")

#     # convert to numpy array
#     token_array = np.array(token_ids, dtype=np.uint16)

#     # save as .bin
#     out_path = os.path.join(folder_path,"tokenized", f"{split_name}.bin")
#     token_array.tofile(out_path)

#     print(f"{split_name}: saved {len(token_array)} tokens → {out_path}")


In [ ]:
run_pipeline()

Finished val


## Creating Context Windows

In [3]:
train_token = np.fromfile(folder_path + '/tokenized'+ "/train.bin", dtype=np.uint16)
len(train_token)

283769406

In [4]:
# Defining the Context size for the Small Language Model. The context size is defined by how complex we want the model to be.

context_size = 512
total_length = (len(train_token) // context_size) * context_size

total_ids = []
for i in range(0, total_length, context_size):
    context = train_token[i : i + context_size]
    total_ids.append(context)

In [5]:
len(total_ids)

554237

# TRANSFORMER

## Token Embedding

In [6]:
vocab_size = 8000
d_model = 384

# --- 2. Create and Initialize the Embedding Table ---
# This creates a list containing 8000 rows. Each row is a list of 384 random decimals.
embedding_table = []
for _ in range(vocab_size):
    # Generate 384 random numbers for a single word profile
    word_vector = [random.uniform(-0.1, 0.1) for _ in range(d_model)]
    embedding_table.append(word_vector)


In [ ]:
import random
import numpy as np
import os

batch_size = 4

def stream_and_save_numpy(data_chunks,batch_size,output_dir):

    os.makedirs(output_dir, exist_ok=True)

    # shuffled_chunks = data_chunks.copy()
    # random.shuffle(shuffled_chunks)

    total_samples = len(data_chunks)

    x_path = os.path.join(output_dir, "X.dat")
    y_path = os.path.join(output_dir, "Y.dat")

    # Create disk-backed arrays
    X_memmap = np.memmap(
        x_path,
        dtype=np.int32,
        mode="w+",
        shape=(total_samples, 511)
    )

    Y_memmap = np.memmap(
        y_path,
        dtype=np.int32,
        mode="w+",
        shape=(total_samples, 511)
    )

    write_pos = 0

    print(f"Processing {total_samples:,} contexts")

    for i in range(0, total_samples, batch_size):

        batch_group = data_chunks[i:i + batch_size]

        if len(batch_group) < batch_size:
            break

        X_batch = np.asarray(
            [window[:-1] for window in batch_group],
            dtype=np.int32
        )

        Y_batch = np.asarray(
            [window[1:] for window in batch_group],
            dtype=np.int32
        )

        end_pos = write_pos + len(batch_group)

        X_memmap[write_pos:end_pos] = X_batch
        Y_memmap[write_pos:end_pos] = Y_batch

        write_pos = end_pos

        if write_pos % 100000 == 0:
            print(
                f"[PROGRESS] {write_pos:,} contexts processed"
            )

    X_memmap.flush()
    Y_memmap.flush()

    print(f"\nFinished.")
    print(f"Saved {write_pos:,} samples")

    return write_pos

In [ ]:
output_dir = folder_path + "/token_embedded"

total_saved = stream_and_save_numpy(
    total_ids,
    batch_size,
    output_dir
)

Processing 554,237 contexts
[PROGRESS] 100,000 contexts processed
[PROGRESS] 200,000 contexts processed
[PROGRESS] 300,000 contexts processed
[PROGRESS] 400,000 contexts processed
[PROGRESS] 500,000 contexts processed

Finished.
Saved 554,236 samples


In [7]:
import numpy as np
output_dir = folder_path + "/token_embedded"
num_samples = 554237

X = np.memmap(
    output_dir + "/X.dat",
    dtype=np.int32,
    mode="r",
    shape=(num_samples, 511)
)

x_tensor = torch.from_numpy(X)
x_tensor.shape

Y = np.memmap(
    output_dir + "/Y.dat",
    dtype=np.int32,
    mode="r",
    shape=(num_samples, 511)
)

y_tensor = torch.from_numpy(X)
y_tensor.shape

/tmp/ipykernel_2524/1332193832.py:12: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x_tensor = torch.from_numpy(X)


torch.Size([554237, 511])

In [8]:
vocab_size = 8000
d_model = 384

embedding_table = []
for _ in range(vocab_size):
    # Generate 384 random numbers for a single word profile
    word_vector = [random.uniform(-0.1, 0.1) for _ in range(d_model)]
    embedding_table.append(word_vector)

lookup_table = np.array(embedding_table, dtype=np.float32)

: 

: 

: 